# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 5: Fine-tuning a Frontier Model

Now we will use OpenAI's API to fine-tune our own private variant of GPT-4.1-nano

In [40]:
# imports

import os
import re
import json
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from pricer.items  import Item
from pricer.evaluator import evaluate
import plotly.io as pio
from getpass import getpass


pio.renderers.default = "notebook_connected"

In [2]:
os.environ['OPENAI_API_KEY'] = getpass("Enter your OPENAI_API_KEY: ")

Enter your OPENAI_API_KEY:  ········


In [3]:
# environment

LITE_MODE = False

load_dotenv(override=True)
# hf_token = os.environ['HF_TOKEN']
# login(hf_token, add_to_git_credential=True)

login()

In [4]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 800,000 training items, 10,000 validation items, 10,000 test items


In [5]:
openai = OpenAI()

# Data size

OpenAI recommends fine-tuning with a small population of 50-100 examples

I'm going to go with 20,000 points.

This cost me $3.42 - you should stick with 100 examples and the cost will be minimal!

In [7]:
# OpenAI recommends fine-tuning with populations of 50-100 examples
# But as our examples are very small, I'm suggesting we go with 100 examples (and 1 epoch)
# fine_tune_train = train[:100]
# fine_tune_validation = val[:50]

## Here I just desided to go with 20000 examples
fine_tune_train = train[:20_000]
fine_tune_validation = val[:100]

In [8]:
len(fine_tune_train)

20000

# Step 1

Prepare our data for fine-tuning in JSONL (JSON Lines) format and upload to OpenAI

In [9]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]

In [10]:
messages_for(fine_tune_train[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single‑piece oil‑rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4" minimum center‑to‑center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation.'},
 {'role': 'assistant', 'content': '$64.30'}]

In [11]:
# Convert the items into a list of json objects - a "jsonl" string
# Each row represents a message in the form:
# {"messages" : [{"role": "system", "content": "You estimate prices...


def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str +'}\n'
    return result.strip()

In [12]:
print(make_jsonl(train[:3]))

{"messages": [{"role": "user", "content": "Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single\u2011piece oil\u2011rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4\" minimum center\u2011to\u2011center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation."}, {"role": "assistant", "content": "$64.30"}]}
{"messages": [{"role": "user", "content": "Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Mini Electric Air Duster Fan  \nCategory: Electronics  \nBrand: Kica  \nDescription: Ultra\u2011compact 86,000\u202fRPM electric air duster with 11\u202fm/s wind speed for precise cleaning and inflation.  \nDetails: Powered by a 9.99\u202fWh motor, adjustable in four speed levels, it uses three 

In [13]:
# Convert the items into jsonl and write them to a file

def write_jsonl(items, filename):
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)

In [14]:
write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")

In [15]:
write_jsonl(fine_tune_validation, "jsonl/fine_tune_validation.jsonl")

In [16]:
with open("jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

In [17]:
train_file

FileObject(id='file-JYWUAvZuST6bj7pLYEeRNG', bytes=11024890, created_at=1772630092, filename='fine_tune_train.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [18]:
with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

In [19]:
validation_file

FileObject(id='file-Rtey24XK4jsxD5fBKK81n8', bytes=55812, created_at=1772630098, filename='fine_tune_validation.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

https://platform.openai.com/storage/files/

# Step 2

## And now time to Fine-tune!

In [20]:
openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 16}, # batch_size 1 for small datasets i.e 100 examples
    suffix="pricer"
)

FineTuningJob(id='ftjob-xQaTPxQoaFCbuW3IJIOIs8ed', created_at=1772630374, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=16, learning_rate_multiplier='auto', n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-rIjz030bRAOFDxPLPMmpFzx4', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-JYWUAvZuST6bj7pLYEeRNG', validation_file='file-Rtey24XK4jsxD5fBKK81n8', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=16, learning_rate_multiplier='auto', n_epochs=1))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=None)

In [21]:
openai.fine_tuning.jobs.list(limit=1)

SyncCursorPage[FineTuningJob](data=[FineTuningJob(id='ftjob-xQaTPxQoaFCbuW3IJIOIs8ed', created_at=1772630374, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=16, learning_rate_multiplier='auto', n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-rIjz030bRAOFDxPLPMmpFzx4', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-JYWUAvZuST6bj7pLYEeRNG', validation_file='file-Rtey24XK4jsxD5fBKK81n8', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=16, learning_rate_multiplier='auto', n_epochs=1))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=None)], has_more=False, object=

In [22]:
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id

In [23]:
job_id

'ftjob-xQaTPxQoaFCbuW3IJIOIs8ed'

In [32]:
openai.fine_tuning.jobs.retrieve(job_id)

FineTuningJob(id='ftjob-xQaTPxQoaFCbuW3IJIOIs8ed', created_at=1772630374, error=Error(code=None, message=None, param=None), fine_tuned_model='ft:gpt-4.1-nano-2025-04-14:sunbird-ai:pricer:DFgt6q5c', finished_at=1772632151, hyperparameters=Hyperparameters(batch_size=16, learning_rate_multiplier=0.1, n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-rIjz030bRAOFDxPLPMmpFzx4', result_files=['file-N4dyA3hGcW4JdA6UVbX4nf'], seed=42, status='succeeded', trained_tokens=2270081, training_file='file-JYWUAvZuST6bj7pLYEeRNG', validation_file='file-Rtey24XK4jsxD5fBKK81n8', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=16, learning_rate_multiplier=0.1, n_epochs=1))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_e

In [31]:
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

[FineTuningJobEvent(id='ftevent-dml7tK5ouXVXZS4LJJ06zUwm', created_at=1772632880, level='info', message='The job has successfully completed', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-PamalvOboxNLGKtgEeTsMVZ9', created_at=1772632877, level='info', message='Usage policy evaluations completed, model is now enabled for sampling', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-5mFxfumGvbM1NdP4tpjflrDS', created_at=1772632877, level='info', message='Moderation checks for snapshot ft:gpt-4.1-nano-2025-04-14:sunbird-ai:pricer:DFgt6q5c passed.', object='fine_tuning.job.event', data={'blocked': False, 'results': [{'flagged': False, 'category': 'harassment/threatening', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'sexual', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'sexual/minors', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'propaganda', 'enforcement': 'blocki

https://platform.openai.com/finetune


# Step 3

Test our fine tuned model

In [33]:
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

In [34]:
fine_tuned_model_name

'ft:gpt-4.1-nano-2025-04-14:sunbird-ai:pricer:DFgt6q5c'

In [35]:
# The prompt

def test_messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
    ]

In [36]:
# Try this out

test_messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [37]:
# The inference function


def gpt_4__1_nano_fine_tuned(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=7
    )
    return response.choices[0].message.content

In [38]:
print(test[0].price)
print(gpt_4__1_nano_fine_tuned(test[0]))

219.0
$269.00


In [41]:
evaluate(gpt_4__1_nano_fine_tuned, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$40 $213 $20 $40 $170 $180 $106 $6 $0 $100 $513 $80 $1 $9 $6 $21 $6 $6 $21 $25 $13 $68 $125 $221 $103 $254 $306 $3 $29 $53 $50 $4 $40 $8 $60 $50 $61 $65 $91 $12 $162 $40 $3 $45 $80 $3 $121 $13 $80 $39 $26 $103 $183 $30 $39 $164 $4 $80 $212 $4 $31 $28 $55 $13 $389 $10 $80 $324 $74 $53 $13 $24 $171 $16 $28 $15 $55 $0 $13 $1 $70 $3 $4 $59 $2 $15 $25 $143 $10 $30 $18 $31 $1 $16 $0 $138 $6 $221 $50 $365 $10 $16 $3 $31 $114 $65 $10 $201 $2 $71 $36 $658 $84 $102 $80 $155 $70 $23 $24 $177 $14 $100 $2 $17 $15 $20 $5 $51 $112 $11 $90 $18 $24 $11 $110 $1 $30 $20 $55 $4 $14 $75 $29 $39 $23 $65 $91 $135 $67 $13 $2 $104 $27 $5 $7 $70 $44 $40 $62 $34 $310 $13 $0 $9 $452 $0 $122 $20 $3 $15 $21 $11 $210 $11 $41 $79 $109 $132 $28 $37 $236 $75 $138 $91 $43 $7 $78 $27 $31 $0 $58 $26 $18 $61 $21 $65 $19 $166 $18 $14 

In [ ]:
# 96.58 - mini 200
# 79.29 - mini 2000
# 82.26 - nano 2000
# 67.75 - nano 20,000